In [13]:
from dotenv import load_dotenv
load_dotenv()

True

## STEP 0 - Load a document ##

In [1]:
# STEP 0- loading a document to be chunked for embedding and vector storage
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("NovaS.pdf")
documents = loader.load()
documents


C:\Users\jethw\AppData\Local\Temp\ipykernel_26012\976360947.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


ImportError: cannot import name '__version__' from 'pydantic_core' (unknown location)

In [15]:
#STEP 1 - splitting the document into chunks using text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents(documents)
len(chunks)

101

In [ ]:
#STEP 2 & 3 - Create embeddings and store them in a vector database
from langchain_openai import OpenAIEmbeddings
#model="text-embedding-3-small" is the latest embedding model from OpenAI, 
# which is more cost-effective and faster than the previous models. It is suitable for most use cases, 
# including semantic search and clustering.
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
#pass the list of chunks to the embed_documents method to get the embeddings for each chunk i.e. the vectror representation of each chunk. 
# The output is a list of embeddings, where each embedding corresponds to a chunk in the same order as the input list.
chunk_embeddings = embed_model.embed_documents([i.page_content for i in chunks]) 
len(chunk_embeddings)


101

In [ ]:
#STEP 2 & 3 - Create embeddings and store them in a vector database
from langchain_community.vectorstores import Chroma
# Create a Chroma vector store from the chunk embeddings and the corresponding chunks
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embed_model,
    persist_directory="./chroma_db"
)

In [16]:
## Step 4 - Connection to the vector database and retrieval of relevant chunks based on a query
vectorstore_conn = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


In [19]:
## Step 5 - Create a retriever from the vectorstore connection and use it to retrieve relevant chunks based on a query
# retriever = vectorstore_conn.as_retriever(search_type="similarity", search_kwargs={"k": 2})
#do similarity search on the vectorstore connection using the query "What is NovaS?" and retrieve the top 2 most similar chunks
query = "When was NovaSphere Technologies was founded?"
retrieved_chunks = vectorstore_conn.similarity_search(query, k=2)
retrieved_chunks

[Document(metadata={'creationdate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'creator': 'Microsoft® Word for Microsoft 365', 'producer': 'Microsoft® Word for Microsoft 365', 'page': 0, 'page_label': '1', 'moddate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba'}, page_content='In 2018, NovaSphere Technologies started getting more attention in the local technology'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'page': 2, 'moddate': '2026-03-31T11:24:15-03:00', 'page_label': '3', 'creator': 'Microsoft® Word for Microsoft 365', 'source': 'NovaS.pdf', 'total_pages': 3, 'author': 'Ansh Lamba', 'creationdate': '2026-03-31T11:24:15-03:00'}, page_content='Today, NovaSphere Technologies is considered a reliable organization that provides data')]

In [33]:
## Step 6 - LLM Integration and Answer Generation
# Import the necessary libraries for LLM integration and answer generation. Use the retrieved chunks to generate a response to the query using a language model.
from langchain_openai import ChatOpenAI
llm_model = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

#form the query which yoou want to ask the LLM model. 
query = input() #"By 2019, how many employees were there?"

#retrieve the relevant chunks based on the query using the vectorstore connection. The retrieved chunks will be used as context for the LLM model to generate a response.
relevant_chunks = vectorstore_conn.similarity_search(query, k=3)

#fetch the context from the retrieved chunks to provide to the LLM model for generating a response. Fetch from the page_content of each chunk.
context = "\n".join([chunk.page_content for chunk in relevant_chunks])
print(f"Context: {context}")
# Pass the query and the context to the LLM model to generate a response. The LLM model will use the context to generate a response to the query.
llm_response = llm_model.invoke(f"{query}, answer the question based on the context provided. Context: {context}")
llm_response
# llm_model.invoke(f"config={{'context': '{context}', 'query': '{query}'}}")


Context: By the beginning of 2019, the organization had grown to more than fifteen employees. This
During 2023, the organization experienced steady growth. The number of employees
number of employees increased again, and the company also started hiring people who


AIMessage(content=' By 2019, there were more than fifteen employees in the organization.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 79, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ET6PgyBdryysTI1hJTCjw00t4j9Xu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0e85e-0697-7450-a728-a12afde43a22-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 15, 'total_tokens': 94, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})